In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================

In [ ]:
# ===== 공통 유틸: 정규식 패턴 / 날짜 검증 / 키워드 사전 =====
# (원본: regex_filters.py, date_normalizer.py, keyword_dict.py)
from __future__ import annotations

import glob
import re
from dataclasses import dataclass
from datetime import datetime
from typing import List, Optional, Pattern, Tuple

import cv2
import numpy as np
import pandas as pd

# ---- 날짜 후보 정규식 (그룹 순서: 'ymd' 또는 'dmy') ----
# 구분자 문자 클래스에 콤마(,)를 포함: OCR이 마침표(.)를 콤마로 오인식하는 사례 대응
DATE_PATTERNS: List[Tuple[Pattern, str]] = [
    (re.compile(r'(?<!\d)(\d{4})[.\-/,\s](\d{1,2})[.\-/,\s](\d{1,2})(?!\d)'), 'ymd'),
    (re.compile(r'(?<!\d)(\d{1,2})[.\-/,\s](\d{1,2})[.\-/,\s](\d{4})(?!\d)'), 'dmy'),
    (re.compile(r'(?<!\d)(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일'), 'ymd'),
    (re.compile(r'(?<!\d)(\d{4})(\d{2})(\d{2})(?!\d)'), 'ymd'),
    (re.compile(r'(?<!\d)(\d{2})[.\-/,\s](\d{1,2})[.\-/,\s](\d{1,2})(?!\d)'), 'ymd'),
    (re.compile(r'(?<!\d)(\d{1,2})[.\-/,\s](\d{1,2})[.\-/,\s](\d{2})(?!\d)'), 'dmy'),
]

# 날짜가 아닌 값(바코드/전화번호/영양성분)이 섞인 박스는 통째로 후보에서 제외
BLACKLIST_CONTEXT_PATTERNS: List[Pattern] = [
    re.compile(r'(?<!\d)\d{12,13}(?!\d)'),
    re.compile(r'01[016789][-.\s]?\d{3,4}[-.\s]?\d{4}'),
    re.compile(r'\d+\s*(?:kcal|mg|g|ml)', re.IGNORECASE),
]


def is_blacklisted_context(text: str) -> bool:
    return any(pattern.search(text) for pattern in BLACKLIST_CONTEXT_PATTERNS)


NONE_RESULT: Tuple[str, str, str] = ("NONE", "NONE", "NONE")
YEAR_MIN, YEAR_MAX = 2015, 2032


def validate_date(year: str, month: str, day: str) -> Optional[Tuple[str, str, str]]:
    """(year, month, day) 문자열이 실제 존재하는 날짜인지 확인하고 YYYY/MM/DD로 정규화."""
    try:
        y, m, d = int(year), int(month), int(day)
        if y < 100:
            y += 2000
        if not (YEAR_MIN <= y <= YEAR_MAX):
            return None
        datetime(y, m, d)
        return (f"{y:04d}", f"{m:02d}", f"{d:02d}")
    except (ValueError, TypeError):
        return None


POSITIVE_KEYWORDS = ['소비기한', '유통기한', '까지', 'EXP', 'exp', 'Exp', 'Best Before', 'BEST BEFORE']
NEGATIVE_KEYWORDS = [
    '제조일자', '제조년월일', '제조일', '생산일자', '생산일', '제조', 'MFG', 'MFD',
    '품목보고번호', 'LOT', '부터',
]

In [ ]:
# ===== 크롭 이미지 전처리 (원본: image_preprocessing.py) =====
def apply_clahe(img: np.ndarray, clip_limit: float = 2.0, tile_grid_size: tuple = (8, 8)) -> np.ndarray:
    """CLAHE(대비 제한 적응 히스토그램 평활화)를 밝기(L) 채널에만 적용해 대비를 강화한다.
    각인/도트 프린팅처럼 배경과 글자의 밝기 차이가 작은 경우 OCR 인식률 개선에 도움이 된다."""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

In [ ]:
# ===== 후보 판별 로직 (원본: candidate_classifier.py) =====
# OCR 박스들 중에서 진짜 소비기한 날짜를 찾아내는 판별 로직.
# - 페이지 전체에서 날짜 후보를 모두 모으고
# - '소비기한'/'유통기한'/'까지' 같은 앵커 키워드와의 거리로 점수를 매겨 최종 후보를 고른다.
# - OCR이 날짜 하나를 여러 박스로 쪼개서 인식하는 경우를 대비해, 읽는 순서상 인접한
#   박스 2~3개를 이어붙인 '가상 박스'도 후보에 포함시킨다.
import math


@dataclass
class TextBox:
    bbox: list
    text: str
    confidence: float


@dataclass
class DateCandidate:
    box_index: int
    bbox: list
    year: str
    month: str
    day: str
    pattern_rank: int  # DATE_PATTERNS 내 순서 (낮을수록 신뢰도 높음)


def extract_date_candidates(box_index: int, box: TextBox) -> List[DateCandidate]:
    """한 텍스트 박스 안에서 날짜로 해석 가능한 모든 부분 문자열을 찾는다."""
    if is_blacklisted_context(box.text):
        return []

    candidates = []
    claimed_spans: List[Tuple[int, int]] = []

    for rank, (pattern, order) in enumerate(DATE_PATTERNS):
        for match in pattern.finditer(box.text):
            span = match.span()
            if any(span[0] < e and span[1] > s for s, e in claimed_spans):
                continue

            g1, g2, g3 = match.groups()
            if order == 'ymd':
                validated = validate_date(g1, g2, g3)
            else:
                validated = validate_date(g3, g2, g1)

            if validated:
                claimed_spans.append(span)
                candidates.append(DateCandidate(
                    box_index=box_index, bbox=box.bbox,
                    year=validated[0], month=validated[1], day=validated[2],
                    pattern_rank=rank,
                ))
    return candidates


def _levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cost = 0 if ca == cb else 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev = curr
    return prev[-1]


def _fuzzy_contains(text: str, keyword: str, max_dist: int = 1) -> bool:
    """편집거리 max_dist 이내로 비슷한 부분 문자열이 있는지 확인 (짧은 키워드는 정확 일치만 인정)."""
    if keyword in text:
        return True
    klen = len(keyword)
    if klen <= 2:
        return False
    for wlen in range(max(1, klen - max_dist), klen + max_dist + 1):
        if wlen > len(text):
            continue
        for start in range(len(text) - wlen + 1):
            window = text[start:start + wlen]
            if _levenshtein(window, keyword) <= max_dist:
                return True
    return False


def _any_keyword_match(text: str, keywords: List[str]) -> bool:
    return any(_fuzzy_contains(text, kw) for kw in keywords)


def bbox_center(bbox: list) -> Tuple[float, float]:
    xs = [pt[0] for pt in bbox]
    ys = [pt[1] for pt in bbox]
    return (sum(xs) / len(xs), sum(ys) / len(ys))


def bbox_height(bbox: list) -> float:
    ys = [pt[1] for pt in bbox]
    return max(ys) - min(ys)


def bbox_union(bboxes: List[list]) -> list:
    xs = [pt[0] for bbox in bboxes for pt in bbox]
    ys = [pt[1] for bbox in bboxes for pt in bbox]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    return [[x0, y0], [x1, y0], [x1, y1], [x0, y1]]


def sort_boxes_reading_order(boxes: List[TextBox]) -> List[int]:
    if not boxes:
        return []
    heights = [bbox_height(b.bbox) for b in boxes]
    avg_height = (sum(heights) / len(heights)) or 20.0
    line_bucket = max(avg_height * 0.6, 1.0)

    def sort_key(i: int):
        cx, cy = bbox_center(boxes[i].bbox)
        return (round(cy / line_bucket), cx)

    return sorted(range(len(boxes)), key=sort_key)


def build_merged_boxes(boxes: List[TextBox], max_window: int = 3) -> List[TextBox]:
    """OCR이 하나의 날짜를 여러 박스로 쪼개서 인식하는 경우를 대비해,
    읽는 순서상 인접한 박스 2~3개를 이어붙인 '가상 박스'를 추가로 만든다."""
    order = sort_boxes_reading_order(boxes)
    merged: List[TextBox] = []
    for window_size in range(2, max_window + 1):
        for start in range(len(order) - window_size + 1):
            idxs = order[start:start + window_size]
            merged_text = ' '.join(boxes[i].text for i in idxs)
            merged_bbox = bbox_union([boxes[i].bbox for i in idxs])
            merged_conf = min(boxes[i].confidence for i in idxs)
            merged.append(TextBox(bbox=merged_bbox, text=merged_text, confidence=merged_conf))
    return merged


def choose_best_date_with_score(boxes: List[TextBox]) -> Tuple[Tuple[str, str, str], float]:
    """페이지 전체 문맥(앵커 키워드와의 거리)을 이용해 가장 그럴듯한 소비기한과 점수를 반환."""
    boxes = list(boxes) + build_merged_boxes(boxes)

    all_candidates: List[DateCandidate] = []
    for idx, box in enumerate(boxes):
        all_candidates.extend(extract_date_candidates(idx, box))

    if not all_candidates:
        return NONE_RESULT, float('-inf')

    centers = [bbox_center(box.bbox) for box in boxes]
    xs = [c[0] for c in centers]
    ys = [c[1] for c in centers]
    diag = math.hypot((max(xs) - min(xs)) or 1.0, (max(ys) - min(ys)) or 1.0) or 1.0

    positive_centers = [centers[i] for i, box in enumerate(boxes)
                         if _any_keyword_match(box.text, POSITIVE_KEYWORDS)]
    negative_centers = [centers[i] for i, box in enumerate(boxes)
                         if _any_keyword_match(box.text, NEGATIVE_KEYWORDS)]

    def nearest_dist(point, others):
        if not others:
            return None
        return min(math.hypot(point[0] - o[0], point[1] - o[1]) for o in others)

    best_candidate = None
    best_score = float('-inf')

    for cand in all_candidates:
        box = boxes[cand.box_index]
        center = bbox_center(cand.bbox)
        score = 0.0

        same_box_positive = _any_keyword_match(box.text, POSITIVE_KEYWORDS)
        same_box_negative = _any_keyword_match(box.text, NEGATIVE_KEYWORDS)
        if same_box_positive:
            score += 5.0
        if same_box_negative:
            # 제조일자와 소비기한이 월/일은 같고 연도만 어긋나는 제품이 흔해서, 양성 가중치보다
            # 확실히 큰 벌점을 줘서 제조일자류 키워드가 있으면 다른 후보가 없는 한 채택되지 않게 한다.
            score -= 10.0

        if not same_box_positive:
            dist = nearest_dist(center, positive_centers)
            if dist is not None:
                score += max(0.0, 3.0 * (1.0 - dist / diag))
        if not same_box_negative:
            dist = nearest_dist(center, negative_centers)
            if dist is not None:
                score -= max(0.0, 6.0 * (1.0 - dist / diag))

        score += (len(DATE_PATTERNS) - cand.pattern_rank) * 0.1

        if score > best_score:
            best_score = score
            best_candidate = cand

    if best_candidate is None:
        return NONE_RESULT, float('-inf')
    return (best_candidate.year, best_candidate.month, best_candidate.day), best_score

In [ ]:
# ===== OCR 엔진 (원본: predict_v2.py의 BasicOCREngine) =====
# RapidOCR(ONNXRuntime 기반 PP-OCR)을 사용한다 - EasyOCR보다 가볍고 CPU에서 특히 빠르다.
# 기본 인식 모델은 중국어+영어 위주라 한글은 Rec.lang_type=LangRec.KOREAN으로 명시 지정해야
# '소비기한/까지' 같은 앵커 키워드가 정상적으로 인식된다. 다만 KOREAN 모델은 각인/도트
# 프린팅 숫자를 거의 못 읽어서, 같은 크롭을 EN 인식 모델로 한 번 더 돌려 보완한다.
# 검출(det) 모델은 언어와 무관하므로 한 번만 돌리고(det-once), 인식(rec)만 KOREAN/EN
# 두 번 돌려서(rec-twice) 중복 검출 비용을 없앤다.
USE_GPU = False  # 채점 서버는 GPU가 없는 CPU 전용 환경

from rapidocr import LangRec, RapidOCR
from rapidocr.ch_ppocr_rec import TextRecInput


class BasicOCREngine:
    def __init__(self, gpu: bool = USE_GPU):
        self.reader = RapidOCR(params={"Rec.lang_type": LangRec.KOREAN})
        self.reader_digits = RapidOCR(params={"Rec.lang_type": LangRec.EN})

        # RapidOCR 기본값은 limit_type='min'이라 "짧은 변이 736보다 작으면 확대"한다.
        # 우리 크롭은 소비기한 텍스트 영역이라 가로로 길고 세로로 얇아서, 짧은 변(세로) 기준으로
        # 강제로 2.5~7배 확대돼 검출 비용이 크롭 크기와 무관하게 항상 커지는 원인이었다(실측:
        # 1.5~3.2초). limit_type='max'로 바꾸면 "긴 변이 limit_side_len을 넘으면 축소"가 되는데,
        # 우리 크롭은 이미 MAX_CROP_SIDE(900)로 긴 변을 제한해뒀으므로 960을 주면 사실상
        # 리사이즈가 거의 없어져 강제 확대가 사라진다(실측: 검출 비용 ~0.15초로 20배 가까이 감소).
        self.reader.text_det.limit_side_len = 960
        self.reader.text_det.limit_type = "max"

    @staticmethod
    def _dedupe_boxes(boxes: List[TextBox]) -> List[TextBox]:
        """KOREAN+EN 두 패스가 같은 위치에서 완전히 똑같은 텍스트를 읽어내는 경우(숫자만 있는
        박스는 두 모델 다 정확히 맞히는 일이 흔함) 중복 박스가 생긴다. 이걸 그대로 두면
        build_merged_boxes()의 '인접 박스 합치기' 윈도우가 중복 때문에 밀려서 엉뚱한 조합이
        먼저 완성되는 버그가 있었다. 같은 위치에서 같은 텍스트가 나오면 하나만 남긴다."""
        seen = set()
        deduped = []
        for box in boxes:
            cx = sum(pt[0] for pt in box.bbox) / len(box.bbox)
            cy = sum(pt[1] for pt in box.bbox) / len(box.bbox)
            key = (box.text, round(cx), round(cy))
            if key in seen:
                continue
            seen.add(key)
            deduped.append(box)
        return deduped

    def read_array(self, img: np.ndarray) -> List[TextBox]:
        """이미 메모리에 있는 (크롭된) 이미지 배열을 OCR한다."""
        try:
            reader = self.reader
            raw_h, raw_w = img.shape[:2]

            proc_img, ratio_h, ratio_w = reader.preprocess(img)
            op_record = {"preprocess": {"ratio_h": ratio_h, "ratio_w": ratio_w}}
            proc_img, op_record = reader.maybe_add_letterbox(proc_img, op_record)

            det_res = reader.text_det(proc_img)
            if det_res.boxes is None:
                return []

            crops = reader.get_crop_img_list(proc_img, det_res)
            cls_res = reader.text_cls(crops)
            crops = cls_res.img_list

            origin_boxes = reader._get_origin_points(det_res.boxes, op_record, raw_h, raw_w)
            text_score_th = reader.text_score

            def to_textboxes(rec_result) -> List[TextBox]:
                out = []
                for box, txt, score in zip(origin_boxes, rec_result.txts, rec_result.scores):
                    if float(score) >= text_score_th:
                        out.append(TextBox(bbox=[[float(pt[0]), float(pt[1])] for pt in box],
                                            text=txt, confidence=float(score)))
                return out

            korean_rec = reader.text_rec(TextRecInput(img=list(crops), return_word_box=False))
            boxes = to_textboxes(korean_rec)

            # KOREAN 패스만으로 이미 4자리 연도 날짜 패턴이 뚜렷하게 읽혔다면 그 결과를 신뢰하고
            # 두 번째 인식(rec) 호출을 건너뛴다 (인쇄 상태가 좋은 흔한 경우 - 속도 절감).
            combined_text = " ".join(box.text for box in boxes)
            has_confident_date = any(pattern.search(combined_text) for pattern, _order in DATE_PATTERNS[:2])
            if not has_confident_date:
                en_rec = self.reader_digits.text_rec(TextRecInput(img=list(crops), return_word_box=False))
                boxes += to_textboxes(en_rec)

            return self._dedupe_boxes(boxes)
        except Exception:
            return []

In [ ]:
# ===== YOLO 위치 탐지 + 크롭 (원본: predict_v2.py) =====
# 500장을 40분 안에 처리해야 하는 시간 예산 때문에, 이미지 전체를 OCR로 훑는 대신
# YOLOv8로 '소비기한 텍스트 영역'만 먼저 찾고 그 작은 크롭만 OCR한다.
MAX_CANDIDATE_BOXES = 3        # confidence 상위 몇 개 박스까지 크롭+OCR을 시도할지.
                                # 227장 실측: 1개(18.9분 환산), 2개(28.1분, 47.1%), 3개(33.3분,
                                # 48.9%) - 3개가 예산(40분) 안에서 가장 정확도가 높아 확정.
EARLY_EXIT_SCORE = 3.0         # 이 점수 이상인 후보를 찾으면 남은 후보 박스는 보지 않고 채택.
CROP_PADDING_RATIO = 0.4       # 크롭 시 박스 주변 여유
MIN_CROP_SIDE = 640            # 크롭의 긴 변이 이보다 작으면 업스케일 (작은 각인 숫자 인식률 개선)
MAX_CROP_SIDE = 900            # 크롭의 긴 변이 이보다 크면 축소 (처리시간 편차 감소)
DEFAULT_IMGSZ = 640            # YOLO 탐지 해상도 (학습 해상도와 일치)
DEFAULT_CONF = 0.1             # YOLO 탐지 confidence threshold (재현율 우선)


def load_image(path: str) -> Optional[np.ndarray]:
    img_array = np.fromfile(str(path), np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_COLOR)


def crop_with_padding(img: np.ndarray, xyxy: Tuple[float, float, float, float]) -> np.ndarray:
    h, w = img.shape[:2]
    x0, y0, x1, y1 = xyxy
    pad_w = (x1 - x0) * CROP_PADDING_RATIO
    pad_h = (y1 - y0) * CROP_PADDING_RATIO
    x0 = max(0, int(x0 - pad_w))
    y0 = max(0, int(y0 - pad_h))
    x1 = min(w, int(x1 + pad_w))
    y1 = min(h, int(y1 + pad_h))
    return img[y0:y1, x0:x1]


def upscale_if_small(crop: np.ndarray, min_side: int = MIN_CROP_SIDE, max_side: int = MAX_CROP_SIDE) -> np.ndarray:
    """크롭의 긴 변을 [min_side, max_side] 범위 안으로 맞춘다."""
    h, w = crop.shape[:2]
    long_side = max(h, w)
    if long_side == 0:
        return crop
    if long_side < min_side:
        scale = min_side / long_side
        return cv2.resize(crop, (int(round(w * scale)), int(round(h * scale))), interpolation=cv2.INTER_CUBIC)
    if long_side > max_side:
        scale = max_side / long_side
        return cv2.resize(crop, (int(round(w * scale)), int(round(h * scale))), interpolation=cv2.INTER_AREA)
    return crop


def detect_expiry_regions(yolo_model, img, imgsz, conf_threshold, max_boxes):
    """confidence 상위 max_boxes개까지의 소비기한 영역 박스(xyxy)를 반환."""
    results = yolo_model.predict(img, imgsz=imgsz, conf=conf_threshold, verbose=False)
    candidates = []
    for r in results:
        for box in r.boxes:
            candidates.append((float(box.conf[0]), tuple(box.xyxy[0].tolist())))
    candidates.sort(key=lambda item: item[0], reverse=True)
    return [xyxy for _conf, xyxy in candidates[:max_boxes]]

In [ ]:
# ===== 추론 실행 =====
from ultralytics import YOLO

WEIGHTS_PATH = "./weights/best.pt"
IMAGE_EXTS = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')

yolo_model = YOLO(WEIGHTS_PATH)
ocr_engine = BasicOCREngine()

image_files = []
for ext in IMAGE_EXTS:
    image_files.extend(glob.glob(os.path.join(INPUT_DIR, ext)))
image_files = sorted(set(image_files))

print(f"총 {len(image_files)}장 추론 시작")

rows = []
for path in image_files:
    image_id = os.path.splitext(os.path.basename(path))[0]
    year, month, day = NONE_RESULT

    img = load_image(path)
    if img is not None:
        candidate_boxes = detect_expiry_regions(yolo_model, img, DEFAULT_IMGSZ, DEFAULT_CONF, MAX_CANDIDATE_BOXES)
        best_score = float('-inf')
        for xyxy in candidate_boxes:
            crop = crop_with_padding(img, xyxy)
            crop = upscale_if_small(crop)
            crop = apply_clahe(crop)
            boxes = ocr_engine.read_array(crop)
            candidate_date, score = choose_best_date_with_score(boxes)
            if candidate_date != NONE_RESULT and score > best_score:
                best_score = score
                year, month, day = candidate_date
            if best_score >= EARLY_EXIT_SCORE:
                break

    if (year, month, day) == NONE_RESULT:
        final_date = "NONE"
    else:
        final_date = f"{year}-{month}-{day}"

    rows.append({"image_id": image_id, "year": year, "month": month, "day": day, "final_date": final_date})

df = pd.DataFrame(rows, columns=["image_id", "year", "month", "day", "final_date"])
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")